In [ ]:
import subprocess
import sys
from pathlib import Path

result_dir = Path("results_sfl_horizon")
result_dir.mkdir(exist_ok=True)

seeds = range(42, 52)

p_visibles = [0.3, 0.4, 0.5, 0.6, 0.7]

loss_types = ["mse"]

input_len = 12
pred_lens = [3]


methods = (
    "ar,"
    "sub_lp,dk_lp,rg_lp,rg_alg,kron_lp,union_lp,"
    "sffa"
)



failed = []

for loss_type in loss_types:
    output_csv = result_dir / f"horizon_{loss_type}.csv"

    cmd = [
        sys.executable, "sfl_main_horizon.py",

        
        "--seeds", ",".join(str(s) for s in seeds),
        "--p-visibles", ",".join(str(p) for p in p_visibles),

        
        "--input-len", str(input_len),
        "--pred-lens", ",".join(str(h) for h in pred_lens),

        
        "--split-ratios", "0.7,0.1,0.2",

        
        "--num-timesteps", "0",

        
        "--optimizer", "lbfgs",
        "--lbfgs-max-iter", "10",
        "--epochs", "50",
        "--lr", "1.0",

        "--loss-type", loss_type,

        
        "--methods", methods,

        
        "--k", "10",
        "--r", "2",

        
        "--ridge", "0.",

        
        "--sfl-ridge", "0.",

        
        "--kron-ridge", "0.0",
        "--no-kron-use-pinv",

        "--drop-gap-windows",

        "--output-csv", str(output_csv),
    ]

    print("\n" + "=" * 100)
    print(f"Running loss_type={loss_type}")
    print(" ".join(cmd))
    print("=" * 100 + "\n")

    try:
        with subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)

            return_code = proc.wait()

        if return_code != 0:
            failed.append((loss_type, return_code))
            print(f"FAILED: loss_type={loss_type}, return_code={return_code}")

    except Exception as exc:
        failed.append((loss_type, repr(exc)))
        print(f"FAILED: loss_type={loss_type}, error={exc}")

print("Failed runs:", failed)